In [ ]:
#importer les libs
import sys; sys.path.insert(0, '..') #pour importer depuis src/ et config.py
from vectorbtpro import *
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png" #rendre en PNG sinon vscode crash avec les gros json plotly

## 1. Charger les résultats d'optimisation

In [ ]:
#charger le pickle generé par opti.py
#le pickle contient les sharpe de chaque combo pour chaque fenetre (train + test)
CACHE_FILE = '../cache/kc_wfsl_results.pickle'
results = vbt.load(CACHE_FILE)
print(f"{len(results)} resultats chargés")
results.head(10)

## 2. Séparation train / test

In [ ]:
#separer train et test pour les comparer
train_results = results.xs('train', level='set')
test_results = results.xs('test', level='set')

print(f"Train : {len(train_results)} resultats")
print(f"Test : {len(test_results)} resultats")
print(f"NaN dans train : {train_results.isna().sum()} (combos qui ont pas tourné)")
print(f"NaN dans test : {test_results.isna().sum()}")

## 3. Corrélation train / test

In [ ]:
#correlation entre sharpe train et sharpe test
#positif = les bons params en train sont aussi bons en test = pas d'overfitting
#negatif = les bons params en train sont mauvais en test = overfitting
corr = train_results.corr(test_results)
print(f"Correlation train/test : {corr:.4f}")

if corr > 0.5:
    print("Bonne correlation, les resultats generalisent bien")
elif corr > 0:
    print("Correlation faible, risque d'overfitting modéré")
else:
    print("Correlation negative, overfitting probable")

## 4. Heatmap Sharpe médian (train) — ma_window vs atr_mult

In [ ]:
#heatmap sharpe median sur le train
#chaque case = mediane du sharpe sur toutes les fenetres pour cette paire (ma_window, atr_mult)
#on agrege sur atr_window et sl_stop pour voir la tendance generale
median_sharpe = train_results.groupby(level=['ma_window', 'atr_mult']).median()
heatmap_data = median_sharpe.unstack(level='atr_mult') #pivot pour avoir atr_mult en colonnes

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=[f'{x:.1f}' for x in heatmap_data.columns],
    y=heatmap_data.index,
    colorscale='RdYlGn', #rouge = mauvais, vert = bon
    colorbar=dict(title='Sharpe median')
))
fig.update_layout(
    title='Sharpe median cross-fenetres (train)',
    xaxis_title='atr_mult', yaxis_title='ma_window', height=500
)
fig.show()

## 5. Heatmap différence test - train (overfitting check)

In [ ]:
#heatmap de la diff entre test et train
#bleu = le test fait MIEUX que le train, bonne generalisation
#rouge = le test fait MOINS bien, overfitting sur ces params
diff = test_results - train_results
median_diff = diff.groupby(level=['ma_window', 'atr_mult']).median()
diff_matrix = median_diff.unstack(level='atr_mult')

fig = go.Figure(data=go.Heatmap(
    z=diff_matrix.values,
    x=[f'{x:.1f}' for x in diff_matrix.columns],
    y=diff_matrix.index,
    colorscale='RdBu', zmid=0, #centré sur 0, bleu positif rouge negatif
    colorbar=dict(title='delta Sharpe')
))
fig.update_layout(
    title='Difference test - train (mediane)',
    xaxis_title='atr_mult', yaxis_title='ma_window', height=500
)
fig.show()

## 6. Recherche de combos robustes

In [ ]:
#trouver les combos robustes
#robuste = sharpe > 0 sur au moins 50% des fenetres de train
param_levels = ['ma_window', 'atr_window', 'atr_mult', 'sl_stop']

#pour chaque combo, compter combien de fenetres ont un sharpe > 0
positive_counts = train_results.groupby(level=param_levels).apply(
    lambda x: (x.dropna() > 0).sum()
)
#sharpe median par combo (agregé sur les fenetres)
median_sharpes = train_results.groupby(level=param_levels).median()

n_splits = results.index.get_level_values('split').nunique() #nb de fenetres
robust = positive_counts[positive_counts >= n_splits * 0.5] #garder que les combos robustes
print(f"{len(robust)} combos robustes sur {len(positive_counts)} total")

if len(robust) > 0:
    #parmi les robustes, prendre celle avec le meilleur sharpe median
    robust_medians = median_sharpes.loc[robust.index]
    best_robust = robust_medians.idxmax()
    final_ma, final_atrw, final_atrm, final_sl = best_robust
    final_ma, final_atrw = int(final_ma), int(final_atrw)
    print(f"\nMeilleure combo robuste :")
    print(f"  ma_window={final_ma}, atr_window={final_atrw}, atr_mult={final_atrm}, sl_stop={final_sl}")
    print(f"  Sharpe median train : {robust_medians.loc[best_robust]:.2f}")
    print(f"  Fenetres positives : {robust.loc[best_robust]}/{n_splits}")
else:
    #fallback si aucune combo robuste, on prend quand meme la meilleure
    best_robust = median_sharpes.idxmax()
    final_ma, final_atrw, final_atrm, final_sl = best_robust
    final_ma, final_atrw = int(final_ma), int(final_atrw)
    print(f"\nAucune combo robuste, fallback sur le meilleur sharpe median :")
    print(f"  ma_window={final_ma}, atr_window={final_atrw}, atr_mult={final_atrm}, sl_stop={final_sl}")
    print(f"  Sharpe median : {median_sharpes.loc[best_robust]:.2f}")

#verifier la perf de cette combo sur le test (données jamais vues)
test_medians = test_results.groupby(level=param_levels).median()
test_val = test_medians.loc[(final_ma, final_atrw, final_atrm, final_sl)]
print(f"  Sharpe median test : {test_val:.2f}")

## 7. Performance sur le test (out-of-sample)

In [ ]:
#perf de la meilleure combo sur chaque fenetre de test (out-of-sample)
#c'est ca le vrai test, est ce que les params marchent sur des données jamais vues
test_perf = test_results.xs(
    (final_ma, final_atrw, final_atrm, final_sl),
    level=param_levels
)
print('Sharpe test par fenetre :')
print(test_perf)
print(f"\nMoyen : {test_perf.mean():.2f}")
print(f"Median : {test_perf.median():.2f}")
print(f"Positifs : {(test_perf > 0).sum()}/{len(test_perf)}") #combien de fenetres sont rentables

## 8. Backtest final sur tout le dataset

In [ ]:
from src.strategies.keltner import run_backtest

#charger les données (meme symbole que dans opti.py)
data = pd.read_csv('../data/raw/HYPEUSDT.csv')
data['date'] = pd.to_datetime(data['date'], unit='ms')
data = data.set_index('date')

#bt sur TOUT le dataset avec les params trouvés
#attention c'est du in-sample car les params viennent de ces memes données
pf_final = run_backtest(data, final_ma, final_atrw, final_atrm, final_sl, size=1)
print(pf_final.stats())

#calculer les bandes pour le plot
ema_f = vbt.MA.run(data['close'], window=final_ma, wtype='Exp').ma.shift(1)
atr_f = vbt.ATR.run(data['high'], data['low'], data['close'], window=final_atrw).atr.shift(1)

#plot du bt avec les bandes du keltner
fig = pf_final.plot()
fig.add_trace(go.Scatter(x=data.index, y=ema_f, name='EMA', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=data.index, y=ema_f + final_atrm * atr_f, name='KC Sup', line=dict(color='green', dash='dash'), opacity=0.5))
fig.add_trace(go.Scatter(x=data.index, y=ema_f - final_atrm * atr_f, name='KC Inf', line=dict(color='red', dash='dash'), opacity=0.5))
fig.show()

## 9. Top 10 des meilleures combos

In [ ]:
#top 10 des meilleures combos par sharpe median sur le train
#on ajoute le sharpe test pour voir si ca tient ou si c'est de l'overfitting
top10 = median_sharpes.sort_values(ascending=False).head(10)
top10_df = top10.reset_index()
top10_df.columns = ['ma_window', 'atr_window', 'atr_mult', 'sl_stop', 'sharpe_train']

#recuperer le sharpe test correspondant a chaque combo du top 10
top10_df['sharpe_test'] = [test_medians.loc[tuple(row[:-1])] for _, row in top10_df.iterrows()]
top10_df['delta'] = top10_df['sharpe_test'] - top10_df['sharpe_train'] #delta negatif = overfitting

print("Top 10 combos (train) avec perf test :")
top10_df